# E1 matched evidence: ICL vs QLoRA finetuning on the frozen seed-7 evidence

Question. When a small model gets the exact same evidence about a changed
environment, does it answer the four frozen probes better with the evidence
in its prompt (in-context learning, ICL) or absorbed into its weights
(finetuning, FT)?

The one rule of this experiment. Both arms consume the identical evidence
string, stored in the single variable `EVIDENCE_TEXT`. The ICL arm reads it
in the prompt. The FT arm trains on it as plain language-modeling text and
then answers the probes with an empty context. "From Word to World" never
matched evidence like this, which is the gap E1 fills.

Preregistered predictions (⚠️ lock these with the team before the real run,
current wording is a draft):
1. Detection: FT at or above ICL at 1.5B. Grounds: the Othello result
   (finetuning lifted structure use, ICL plateaued) and the PI comment that
   ICL showed no benefit below 70B in From Word to World.
2. Localization: near floor for both on the stochastic instance, better on
   the deterministic one. Grounds: the frozen pilot, localization 0 of 3
   models on stochastic.
3. Preservation: high for both arms.
4. Adaptation: no confident prediction, treated as exploratory. Finetuning
   on new facts can be slow and can encourage guessing (Gekhman et al.
   2024), so FT could win detection and still lose route planning.

Hardware. Colab T4 is enough. Qwen2.5-1.5B-Instruct in 4-bit with QLoRA.
QLoRA means low-rank adapters trained on top of a 4-bit quantized model.
All data stays in memory. Nothing is written to Drive.

Status. The evidence generator and the scorers below are tested. The model
cells are written but were not executed in this session (⚠️ no GPU here),
so expect the usual first-run friction.

In [ ]:
%pip install -q -U transformers peft bitsandbytes accelerate datasets

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# Model ladder, one model per Colab session, change MODEL_NAME and Run all:
#   "Qwen/Qwen2.5-1.5B-Instruct"   default, ungated
#   "Qwen/Qwen2.5-3B-Instruct"     the size ladder point, ungated
#   "google/gemma-3-4b-it"         family match to the agentic Ollama runs.
#       Gated on Hugging Face, accept the license once, then in Colab:
#       from huggingface_hub import login; login()
SEED = 7
EPOCHS = 20          # the corpus is tiny, many passes are intended
LR = 2e-4
LORA_R, LORA_ALPHA = 16, 32
MAX_NEW_TOKENS = 300

In [ ]:
# Tested core: world reconstruction, balanced evidence collector,
# F2-style rendering, the four probes, local scorers.
"""Pure-python core for the E1 matched-evidence notebook. Tested locally,
then embedded verbatim in the notebook. No model code here."""

from __future__ import annotations

import json
import random

ADJ = {
    "A": {"a1": "D", "a2": "G"},
    "B": {"a1": "D", "a2": "F"},
    "C": {"a1": "E", "a2": "D"},
    "D": {"a1": "A", "a2": "B"},
    "E": {"a1": "C", "a2": "A"},
    "G": {"a1": "A", "a2": "C", "a3": "H"},
    "H": {"a1": "B", "a2": "C"},
}
START, GOAL, BREAK = "E", "F", ("D", "a2")
LEGAL = [(u, a) for u in sorted(ADJ) for a in sorted(ADJ[u])]


def step(node, action, broken, rng, p=None):
    """(status, next_node). Deterministic unless p is given."""
    prob = 0.0 if (node, action) == broken else (
        1.0 if p is None else p[(node, action)])
    if rng.random() < prob:
        return "OK", ADJ[node][action]
    return "DROP", node


def collect_balanced(k=5, broken=None, horizon=60, cap=300, seed=7):
    """Mimic of the frozen collect_balanced: random starts, least-tried
    listed action, until every legal pair has at least k attempts.
    Returns (triples, n_episodes). A triple is (node, action, next_node)."""
    rng = random.Random(seed)
    counts = {pair: 0 for pair in LEGAL}
    triples, episodes = [], 0
    while min(counts.values()) < k and episodes < cap:
        episodes += 1
        node = rng.choice([n for n in ADJ if n != GOAL])
        for _ in range(horizon):
            menu = sorted(ADJ[node])
            least = min(counts[(node, a)] for a in menu)
            action = rng.choice([a for a in menu
                                 if counts[(node, a)] == least])
            status, nxt = step(node, action, broken, rng)
            counts[(node, action)] += 1
            triples.append((node, action, nxt))
            node = nxt
            if node == GOAL:
                break
    return triples, episodes


def render_f2_shuffled(pre_triples, post_triples, seed=7):
    """F2 rendering: bare shuffled (state, action, next state) triples,
    one period section each. Stand-in wording. For the official E1 run,
    replace the output of this function with the archived prompt payload
    from the frozen pilot artifacts so the string is byte identical to
    the offline ICL arm."""
    rng = random.Random(seed)
    pre = [f"({u}, {a}, {v})" for (u, a, v) in pre_triples]
    post = [f"({u}, {a}, {v})" for (u, a, v) in post_triples]
    rng.shuffle(pre)
    rng.shuffle(post)
    return ("Observations, period 1 (order shuffled):\n" + "\n".join(pre)
            + "\n\nObservations, period 2 (order shuffled):\n"
            + "\n".join(post) + "\n")


# -- probes with the frozen answer schemas ---------------------------------

QUERIED_PAIRS = ["H a2", "D a2", "D a1", "C a1"]  # frozen pilot set

PROBES = {
    "detection": ('Between period 1 and period 2, did the network dynamics '
                  'change? Reply with exactly one JSON object: '
                  '{"changed": true} or {"changed": false}.'),
    "localization": ('Exactly one state-action pair changed between the '
                     'periods. Name it. Reply {"node": "...", '
                     '"action": "..."}.'),
    "preservation": ('For each pair below, say whether its behavior changed '
                     'between the periods. Reply one JSON object mapping '
                     f'the pair to "changed" or "unchanged": {QUERIED_PAIRS}'),
    "adaptation": (f'Using only period 2 behavior, give the lowest-cost '
                   f'route from {START} to {GOAL}. Reply '
                   '{"route": [{"node": "...", "action": "..."}, ...]} '
                   'with at most 32 steps.'),
}

GOLD = {
    "detection": {"changed": True},
    "localization": {"node": "D", "action": "a2"},
    "preservation": {"H a2": "unchanged", "D a2": "changed",
                     "D a1": "unchanged", "C a1": "unchanged"},
}


def extract_last_json(text):
    depth, start, last = 0, None, None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth:
            depth -= 1
            if depth == 0:
                last = text[start:i + 1]
    if last is None:
        return None
    try:
        return json.loads(last)
    except json.JSONDecodeError:
        return None


def score(probe, parsed):
    """Local, unofficial scorers mirroring the frozen contract. Official
    numbers come from ecpm_parser.run_probe on commit 5318c3e."""
    if parsed is None:
        return {"status": "malformed", "correct": False}
    if probe == "detection":
        return {"status": "valid",
                "correct": parsed.get("changed") is True}
    if probe == "localization":
        ok = (parsed.get("node") == GOLD["localization"]["node"]
              and parsed.get("action") == GOLD["localization"]["action"])
        return {"status": "valid", "correct": ok}
    if probe == "preservation":
        if isinstance(parsed.get("pairs"), list):
            parsed = {f"{h.get('node')} {h.get('action')}":
                      ("changed" if h.get("changed") else "unchanged")
                      for h in parsed["pairs"]}
        if set(parsed) != set(GOLD["preservation"]):
            return {"status": "invalid_pairs", "correct": False}
        hits = sum(parsed[p] == GOLD["preservation"][p]
                   for p in GOLD["preservation"])
        return {"status": "valid", "correct": hits == 4,
                "hits": f"{hits}/4"}
    if probe == "adaptation":
        route = parsed.get("route") or []
        node, cost = START, 0
        rng = random.Random(0)
        for hop in route[:32]:
            u, a = hop.get("node"), hop.get("action")
            if u != node:
                return {"status": "discontinuous", "correct": False}
            if a not in ADJ.get(u, {}):
                return {"status": "unknown_action", "correct": False}
            status, node = step(u, a, BREAK, rng)
            cost += 1
            if node == GOAL:
                break
        if node != GOAL:
            return {"status": "route_failed", "correct": False}
        regret = cost - 5  # deterministic post-change oracle cost
        return {"status": "valid", "correct": regret == 0,
                "cost": cost, "regret": regret}

## Evidence

The cell below generates stand-in evidence from a local reconstruction of
the seed-7 deterministic world, using a balanced collector that mimics the
frozen `collect_balanced` (random starts, least-tried action, at least k
attempts per legal pair per period).

The official frozen payload is now embedded below and used by default
(USE_OFFICIAL is True). It was extracted from the archived run artifacts
and is byte identical across all four archived official runs. With it,
every probe prompt here reproduces the frozen pilot prompts byte for
byte (shared context, one blank line, question). Set USE_OFFICIAL to
False only to exercise the stand-in generator. Previous instructions,
kept for reference: for a manual swap you would set `EVIDENCE_TEXT`
to the archived prompt payload from the frozen pilot artifacts (Efe's runs
directory, commit 5318c3e), so the string is byte identical to what the
offline ICL arm read. The stand-in exists so the whole pipeline runs today
without the repo.

In [ ]:
# Generate stand-in evidence and run the scorer sanity checks
pre_triples, ep_pre = collect_balanced(k=5, broken=None, seed=7)
post_triples, ep_post = collect_balanced(k=5, broken=BREAK, seed=8)
# Official frozen payload, extracted from the archived run artifacts.
# Identical across all four archived official runs. sha1 is asserted below.
import base64 as _b64, json as _json
OFFICIAL = _json.loads(_b64.b64decode(
    "eyJzaGFyZWRfY29udGV4dCI6ICJZb3UgYXJlIGFuYWx5c2luZyBhIGNvdXJpZXIgbmV0d29yay4gTm9kZXMgYXJlIGxvY2F0aW9uczsgYXQgZWFjaFxubm9kZSB5b3UgbWF5IGF0dGVtcHQgdGhlIGxpc3RlZCBhY3Rpb25zIChhSykuIEFuIGF0dGVtcHQgZWl0aGVyIGRlbGl2ZXJzXG55b3UgdG8gdGhhdCBhY3Rpb24ncyBkZXN0aW5hdGlvbiBvciB5b3Ugc3RheSBhbmQgcmV0cnkgKGVhY2ggYXR0ZW1wdFxuY29zdHMgMSkuIFlvdSBvYnNlcnZlZCB0aGUgbmV0d29yayBpbiB0d28gcGVyaW9kcy5cblxuTm9kZXM6IEEsIEIsIEMsIEQsIEUsIEYsIEcsIEhcblN0YXJ0OiBFICAgR29hbDogRlxuXG5BY3Rpb24gbWVudSwgcGVyaW9kIEEgKGVhcmxpZXIpOiBBOiBhMSwgYTI7IEI6IGExLCBhMjsgQzogYTEsIGEyOyBEOiBhMSwgYTI7IEU6IGExLCBhMjsgRzogYTEsIGEyLCBhMzsgSDogYTEsIGEyXG5BY3Rpb24gbWVudSwgcGVyaW9kIEIgKGxhdGVyKTogQTogYTEsIGEyOyBCOiBhMSwgYTI7IEM6IGExLCBhMjsgRDogYTEsIGEyOyBFOiBhMSwgYTI7IEc6IGExLCBhMiwgYTM7IEg6IGExLCBhMlxuXG5PYnNlcnZhdGlvbnMsIHBlcmlvZCBBOlxuKEEsIGEyLCBHKVxuKEgsIGExLCBCKVxuKEMsIGEyLCBEKVxuKEcsIGEyLCBDKVxuKEEsIGEyLCBHKVxuKEIsIGEyLCBGKVxuKEIsIGExLCBEKVxuKEcsIGExLCBBKVxuKEIsIGExLCBEKVxuKEIsIGExLCBEKVxuKEQsIGExLCBBKVxuKEcsIGEyLCBDKVxuKEQsIGExLCBBKVxuKEUsIGExLCBDKVxuKEQsIGExLCBBKVxuKEMsIGExLCBFKVxuKEMsIGEyLCBEKVxuKEUsIGEyLCBBKVxuKEIsIGEyLCBGKVxuKEgsIGEyLCBDKVxuKEcsIGEzLCBIKVxuKEQsIGEyLCBCKVxuKEUsIGExLCBDKVxuKEcsIGEzLCBIKVxuKEcsIGExLCBBKVxuKEgsIGEyLCBDKVxuKEUsIGExLCBDKVxuKEgsIGExLCBCKVxuKEQsIGExLCBBKVxuKEMsIGExLCBFKVxuKEMsIGExLCBFKVxuKEUsIGExLCBDKVxuKEcsIGExLCBBKVxuKEUsIGEyLCBBKVxuKEUsIGEyLCBBKVxuKEcsIGEzLCBIKVxuKEcsIGEyLCBDKVxuKEQsIGEyLCBCKVxuKEIsIGExLCBEKVxuKEEsIGExLCBEKVxuKEcsIGEzLCBIKVxuKEgsIGExLCBCKVxuKEgsIGEyLCBDKVxuKEgsIGEyLCBDKVxuKEIsIGEyLCBGKVxuKEEsIGEyLCBHKVxuKEcsIGExLCBBKVxuKEEsIGExLCBEKVxuKEgsIGEyLCBDKVxuKEMsIGExLCBFKVxuKEUsIGExLCBDKVxuKEQsIGEyLCBCKVxuKEIsIGExLCBEKVxuKEQsIGEyLCBCKVxuKEgsIGExLCBCKVxuKEcsIGEzLCBIKVxuKEcsIGEyLCBDKVxuKEMsIGEyLCBEKVxuKEMsIGEyLCBEKVxuKEEsIGEyLCBHKVxuKEUsIGEyLCBBKVxuKEMsIGEyLCBEKVxuKEcsIGEyLCBDKVxuKEEsIGEyLCBHKVxuKEQsIGEyLCBCKVxuKEIsIGEyLCBGKVxuKEQsIGExLCBBKVxuKEIsIGEyLCBGKVxuKEcsIGExLCBBKVxuKEEsIGExLCBEKVxuKEEsIGExLCBEKVxuKEgsIGExLCBCKVxuKEUsIGEyLCBBKVxuKEMsIGExLCBFKVxuKEEsIGExLCBEKVxuXG5PYnNlcnZhdGlvbnMsIHBlcmlvZCBCOlxuKEQsIGEyLCBEKVxuKEcsIGEyLCBDKVxuKEQsIGEyLCBEKVxuKEQsIGExLCBBKVxuKEEsIGEyLCBHKVxuKEgsIGExLCBCKVxuKEMsIGEyLCBEKVxuKEgsIGExLCBCKVxuKEIsIGEyLCBGKVxuKEIsIGExLCBEKVxuKEcsIGEyLCBDKVxuKEUsIGEyLCBBKVxuKEcsIGEzLCBIKVxuKEUsIGExLCBDKVxuKEEsIGExLCBEKVxuKEIsIGExLCBEKVxuKEUsIGExLCBDKVxuKEgsIGEyLCBDKVxuKEMsIGExLCBFKVxuKEMsIGEyLCBEKVxuKEUsIGEyLCBBKVxuKEQsIGEyLCBEKVxuKEcsIGEzLCBIKVxuKEgsIGExLCBCKVxuKEMsIGExLCBFKVxuKEQsIGExLCBBKVxuKEcsIGExLCBBKVxuKEMsIGEyLCBEKVxuKEQsIGExLCBBKVxuKEgsIGEyLCBDKVxuKEEsIGExLCBEKVxuKEUsIGEyLCBBKVxuKEIsIGEyLCBGKVxuKEcsIGEzLCBIKVxuKEEsIGEyLCBHKVxuKEIsIGExLCBEKVxuKEEsIGExLCBEKVxuKEMsIGEyLCBEKVxuKEgsIGExLCBCKVxuKEUsIGExLCBDKVxuKEEsIGEyLCBHKVxuKEUsIGEyLCBBKVxuKEMsIGEyLCBEKVxuKEUsIGExLCBDKVxuKEIsIGEyLCBGKVxuKEcsIGExLCBBKVxuKEcsIGEzLCBIKVxuKEcsIGExLCBBKVxuKEgsIGEyLCBDKVxuKEgsIGEyLCBDKVxuKEEsIGEyLCBHKVxuKEEsIGExLCBEKVxuKEQsIGEyLCBEKVxuKEcsIGExLCBBKVxuKEIsIGExLCBEKVxuKEcsIGEyLCBDKVxuKEcsIGEyLCBDKVxuKEUsIGEyLCBBKVxuKEUsIGExLCBDKVxuKEIsIGEyLCBGKVxuKEMsIGExLCBFKVxuKEQsIGEyLCBEKVxuKEMsIGExLCBFKVxuKEEsIGExLCBEKVxuKEIsIGEyLCBGKVxuKEQsIGExLCBBKVxuKEMsIGExLCBFKVxuKEQsIGExLCBBKVxuKEEsIGEyLCBHKVxuKEcsIGEyLCBDKVxuKEcsIGEzLCBIKVxuKEcsIGExLCBBKVxuKEIsIGExLCBEKVxuKEgsIGEyLCBDKVxuKEgsIGExLCBCKSIsICJxdWVzdGlvbnMiOiB7ImRldGVjdGlvbiI6ICJRdWVzdGlvbjogZGlkIHRoZSBuZXR3b3JrJ3MgZHluYW1pY3MgY2hhbmdlIGJldHdlZW4gcGVyaW9kIEEgYW5kIHBlcmlvZCBCP1xuQW5zd2VyIHdpdGggZXhhY3RseSBvbmUgSlNPTiBvYmplY3Q6IHtcImNoYW5nZWRcIjogdHJ1ZX0gb3Ige1wiY2hhbmdlZFwiOiBmYWxzZX0uIE5vIG90aGVyIHRleHQuXG4iLCAibG9jYWxpemF0aW9uIjogIlRoZSBkeW5hbWljcyBjaGFuZ2VkIGJldHdlZW4gdGhlIHBlcmlvZHMuIFF1ZXN0aW9uOiB3aGljaCBzaW5nbGUgKG5vZGUsIGFjdGlvbikgcGFpciBjaGFuZ2VkP1xuQW5zd2VyIHdpdGggZXhhY3RseSBvbmUgSlNPTiBvYmplY3Q6IHtcIm5vZGVcIjogXCI8bm9kZT5cIiwgXCJhY3Rpb25cIjogXCI8YUs+XCJ9LiBObyBvdGhlciB0ZXh0LlxuIiwgInByZXNlcnZhdGlvbiI6ICJGb3IgRUFDSCBvZiB0aGUgZm9sbG93aW5nIChub2RlLCBhY3Rpb24pIHBhaXJzLCBqdWRnZSB3aGV0aGVyIGl0cyBkeW5hbWljcyBjaGFuZ2VkIGJldHdlZW4gcGVyaW9kIEEgYW5kIHBlcmlvZCBCOlxuLSBub2RlIEgsIGFjdGlvbiBhMlxuLSBub2RlIEQsIGFjdGlvbiBhMlxuLSBub2RlIEQsIGFjdGlvbiBhMVxuLSBub2RlIEMsIGFjdGlvbiBhMVxuQW5zd2VyIHdpdGggZXhhY3RseSBvbmUgSlNPTiBvYmplY3Qgb2YgdGhlIGZvcm0ge1wicGFpcnNcIjogW3tcIm5vZGVcIjogXCIuLi5cIiwgXCJhY3Rpb25cIjogXCIuLi5cIiwgXCJjaGFuZ2VkXCI6IHRydWV8ZmFsc2V9LCAuLi5dfSBjb250YWluaW5nIGV2ZXJ5IGxpc3RlZCBwYWlyIGV4YWN0bHkgb25jZS4gTm8gb3RoZXIgdGV4dC5cbiIsICJhZGFwdGF0aW9uIjogIlBsYW4gYSByb3V0ZSBmb3IgcGVyaW9kIEIgKHRoZSBsYXRlciBuZXR3b3JrKSBmcm9tIEUgdG8gRi4gQW5zd2VyIHdpdGggZXhhY3RseSBvbmUgSlNPTiBvYmplY3Qgb2YgdGhlIGZvcm0ge1wicm91dGVcIjogW3tcIm5vZGVcIjogXCIuLi5cIiwgXCJhY3Rpb25cIjogXCIuLi5cIn0sIC4uLl19OiBhdCBtb3N0IDMyIHN0ZXBzLCB0aGUgZmlyc3Qgc3RlcCdzIG5vZGUgbXVzdCBiZSBFLCBlYWNoIG5leHQgc3RlcCdzIG5vZGUgbXVzdCBiZSB3aGVyZSB0aGUgcHJldmlvdXMgYWN0aW9uIGxlYWRzLCBhbmQgdGhlIHJvdXRlIG11c3QgZW5kIGF0IEYuIE5vIG90aGVyIHRleHQuXG4ifSwgInNoYTEiOiAiNGJlZjc2ZTM2NjQ1YWFlYWU4MjBmMWY0OWE0NzJiZDA0NzM4ZTk3NiJ9"
).decode())
print("official payload loaded, preview:")
print(OFFICIAL["shared_context"][:240])

USE_OFFICIAL = True  # False switches back to the stand-in generator

if USE_OFFICIAL:
    EVIDENCE_TEXT = OFFICIAL["shared_context"]
    import hashlib
    assert hashlib.sha1(EVIDENCE_TEXT.encode()).hexdigest() == OFFICIAL["sha1"]
    PROBE_QUESTIONS = OFFICIAL["questions"]
else:
    EVIDENCE_TEXT = render_f2_shuffled(pre_triples, post_triples)
    PROBE_QUESTIONS = PROBES

print(f"pre {len(pre_triples)} triples in {ep_pre} episodes, "
      f"post {len(post_triples)} triples in {ep_post} episodes")
print(f"evidence characters: {len(EVIDENCE_TEXT)}")

assert score("detection", {"changed": True})["correct"]
assert score("localization", {"node": "D", "action": "a2"})["correct"]
good = [{"node": "E", "action": "a2"}, {"node": "A", "action": "a2"},
        {"node": "G", "action": "a3"}, {"node": "H", "action": "a1"},
        {"node": "B", "action": "a2"}]
assert score("adaptation", {"route": good})["regret"] == 0
print("scorer sanity checks passed")

In [ ]:
# Base model in 4-bit and the shared probe runner
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto")

SYSTEM = ("You are analyzing interaction logs from an unknown packet "
          "network. Each observation is (node, action, next node). If the "
          "next node equals the node, the attempt failed and the packet "
          "stayed in place. Answer with exactly one JSON object and "
          "nothing else.")

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def ask(model, question, evidence=None):
    user = (evidence + "\n\n" + question) if evidence else question
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]
    try:
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
    except Exception:
        # some chat templates reject a separate system role, fold it in
        merged = [{"role": "user", "content": SYSTEM + "\n\n" + user}]
        enc = tok.apply_chat_template(merged, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)
    enc = enc.to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)

def run_probes(model, evidence=None, label=""):
    rows = {}
    for name, q in PROBE_QUESTIONS.items():
        raw = ask(model, q, evidence)
        parsed = extract_last_json(raw)
        rows[name] = {"raw": raw, "parsed": parsed, **score(name, parsed)}
    print(label)
    for name, r in rows.items():
        extra = {k: r[k] for k in ("hits", "cost", "regret") if k in r}
        print(f"  {name}: status {r['status']}, correct {r['correct']} {extra}")
    return rows

In [ ]:
# Arm 1, ICL: base weights, evidence in the prompt
icl_results = run_probes(base, evidence=EVIDENCE_TEXT,
                         label="ICL arm (base weights, evidence in context)")

In [ ]:
# Arm 2, FT: QLoRA language-model finetuning on the same string,
# then probes with an EMPTY context
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from datasets import Dataset

ft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto")
ft = prepare_model_for_kbit_training(ft)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                  bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
ft = get_peft_model(ft, lora)

block = 512
ids = tok(EVIDENCE_TEXT)["input_ids"]
chunks = [ids[i:i + block] for i in range(0, len(ids), block)]
ds = Dataset.from_dict({"input_ids": chunks})
collator = DataCollatorForLanguageModeling(tok, mlm=False)
targs = TrainingArguments(
    output_dir="ft_out", num_train_epochs=EPOCHS,
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    learning_rate=LR, logging_steps=5, save_strategy="no",
    report_to="none", seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported())
Trainer(model=ft, args=targs, train_dataset=ds,
        data_collator=collator).train()
ft.eval()
ft_results = run_probes(ft, evidence=None,
    label="FT arm (finetuned weights, no evidence in context)")

In [ ]:
# Arm 3, optional: finetuned weights AND evidence in context
combo_results = run_probes(ft, evidence=EVIDENCE_TEXT,
    label="FT plus ICL arm (finetuned weights, evidence in context)")

In [ ]:
# Persist the raw answers for official re-scoring with rescore_e1.py
import json, hashlib
tag = MODEL_NAME.split("/")[-1]
payload = {"model": MODEL_NAME,
           "evidence_sha1": hashlib.sha1(EVIDENCE_TEXT.encode()).hexdigest(),
           "icl": icl_results, "ft": ft_results, "combo": combo_results}
fname = f"e1_answers_{tag}.json"
json.dump(payload, open(fname, "w"), indent=1)
print("saved", fname, "download it and run rescore_e1.py on it")

In [ ]:
# Results table and the evidence dose
import pandas as pd

ev_tokens = len(tok(EVIDENCE_TEXT)["input_ids"])
ev_transitions = len(pre_triples) + len(post_triples)
rows = []
for arm, res in [("ICL", icl_results), ("FT", ft_results),
                 ("FT+ICL", combo_results)]:
    for probe, r in res.items():
        row = {"arm": arm, "probe": probe, "status": r["status"],
               "correct": r["correct"]}
        row.update({k: r[k] for k in ("hits", "cost", "regret") if k in r})
        rows.append(row)
df = pd.DataFrame(rows)
print(f"evidence dose: {ev_tokens} tokens, {ev_transitions} transitions")
df

## Reporting

What to send Sruthi, numbers first: one table with arm, probe, status, and
correct, plus the evidence dose in tokens and in transitions. The token
count printed above is the number the team needs for the open question of
how to match evidence between the offline and agentic arms.

Re-score the adaptation route with the frozen `ecpm_parser.run_probe` on
commit 5318c3e before any number goes in the doc. The scorers here mirror
the frozen contract but are unofficial.

Extension hook. E2 (evidence ceiling) reuses this notebook unchanged.
Regenerate evidence at k = 5, 10, 15, 20 and watch when localization
starts working, which is exactly Efe's question 1 from the 24/08 check-in.